In [ ]:
!git clone https://github.com/Jun1801/CoDraft-Bench.git

In [ ]:
%cd "YOUR_WORKING_DIR"

In [ ]:
!pip install -r requirements.txt

In [ ]:
%cd D:\Study Documents\Other subjects\CoDraft-Bench
CD_PATH = "YOUR WORKING DIR"

In [ ]:
import os
import sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if CD_PATH not in sys.path:
    sys.path.append(CD_PATH)

In [ ]:
from copy import deepcopy
import numpy as np
import pandas as pd
import torch

from config import *
from preprocess import *
from model import *
from model.models import *
from utils import *

In [ ]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

data_generator = torch.Generator()
data_generator.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
INPUT_ROOT = "dataset"
WORK_DIR = "working"
MODEL_NAME = "microsoft/deberta-v3-base"
MODEL_TYPE = "cross_encoder"
CHECKPOINT_FILE = f"{WORK_DIR}/{MODEL_NAME}_best_model.pth"
PRETRAIN_FILE = None

In [ ]:
tokenizer = get_tokenizer(MODEL_NAME)

In [ ]:
data_manager = DataManager(
    input_root = INPUT_ROOT, 
    work_dir = WORK_DIR, 
    config_data = CONFIG_DATA,
    tokenizer=tokenizer,
    seed_worker = seed_worker,
    data_generator = data_generator,
    random_seed = RANDOM_SEED,
    rebalance=False,
    train_file="train.csv",
    val_file="val.csv",
    test_file="test.csv")

In [ ]:
train_dataloader, evaluator = data_manager.get_dataloaders(model_type=MODEL_TYPE)

In [ ]:
train_df, val_df, test_df = data_manager.get_data()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class_weights = compute_class_weight(train_df['label_score'])
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

In [ ]:
model, _ = get_model(model_type="cross_encoder",model_name=MODEL_NAME, num_classes=5, max_len=256, weights_tensor=weights_tensor)

In [ ]:
trained_model = train_cross_encoder(model, train_dataloader, evaluator)

In [ ]:
test_preds, test_true = get_preds_cross_encoder(model=trained_model, df_test=test_df)

In [ ]:
result_df = pd.DataFrame({
    "label": test_true,
    "pred": test_preds,
})

In [ ]:
result_df.to_csv("result_df.csv", index=False)

In [ ]:
zip_model_folder("./CoDraft-Bench/output")